In [1]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

from transformers import CLIPProcessor, CLIPModel

In [2]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [3]:
import easyocr
from PIL import Image

reader = easyocr.Reader(['en'])

def process_image_pipeline(image_path):
    image = Image.open(image_path).convert("RGB")

    # OCR bằng EasyOCR
    result = reader.readtext("hinhanh.jpg", detail=0)
    ocr_text = " ".join(result)

    return ocr_text

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [4]:
from PIL import Image
import torch
import pytesseract

from transformers import CLIPProcessor, CLIPModel
from transformers import MarianMTModel, MarianTokenizer

In [5]:
def process_image_pipeline(image_path):
    # 1. Mở ảnh
    image = Image.open(image_path).convert("RGB")
    
    # 2. OCR
    ocr_text = pytesseract.image_to_string(image)

    # 3. CLIP
    texts = ["a photo of a building", "a university campus", "a classroom", "a document", "a street view","a photo of a building", 
    "a university campus", 
    "a classroom", 
    "a document", 
    "a street view",
    "the national flag of Vietnam with a yellow star on a red background",
    "the national flag of South Korea",
    "the national flag of the United States",
    "the national flag of Japan",
    "a national flag"]
    
    # Encode ảnh
    image_inputs = clip_processor(images=image, return_tensors="pt")
    image_outputs = clip_model.get_image_features(**image_inputs)
    
    # ÉP KIỂU VỀ TENSOR (Để né lỗi BaseModelOutputWithPooling)
    if not isinstance(image_outputs, torch.Tensor):
        image_features = image_outputs.pooler_output
    else:
        image_features = image_outputs
        
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    
    # Encode chữ
    text_inputs = clip_processor(text=texts, return_tensors="pt", padding=True)
    text_outputs = clip_model.get_text_features(**text_inputs)
    
    if not isinstance(text_outputs, torch.Tensor):
        text_features = text_outputs.pooler_output
    else:
        text_features = text_outputs
        
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # Similarity
    similarity = torch.matmul(image_features, text_features.T)
    caption = texts[similarity.argmax().item()]

    # 4. Dịch thuật
    translated = ""
    if ocr_text.strip():
        inputs = mt_tokenizer(ocr_text, return_tensors="pt", padding=True, truncation=True)
        outputs = mt_model.generate(**inputs)
        translated = mt_tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "ocr_text": ocr_text,
        "translated_text": translated,
        "clip_caption": caption
    }

In [6]:
try:
    # Nhớ kiểm tra file "harvard_0.jpg" có nằm cùng thư mục không nhé
    result = process_image_pipeline("hinhanh.jpg")
    print("--- KẾT QUẢ ---")
    print(f"OCR (Gốc): {result['ocr_text']}")
    print(f"Dịch tiếng Việt: {result['translated_text']}")
    print(f"Dự đoán ảnh: {result['clip_caption']}")
except Exception as e:
    print(f"Vẫn còn lỗi: {e}")

--- KẾT QUẢ ---
OCR (Gốc): 
Dịch tiếng Việt: 
Dự đoán ảnh: the national flag of Vietnam with a yellow star on a red background


In [9]:
import gradio as gr

def gradio_wrapper(input_img):
    # Lưu ảnh tạm thời để hàm cũ của mày đọc được
    temp_path = "temp_ui_image.jpg"
    input_img.save(temp_path)
    
    # Gọi cái hàm "phá đảo" mà mày vừa chạy xong
    res = process_image_pipeline(temp_path)
    
    return res['ocr_text'], res['translated_text'], res['clip_caption']

# Thiết lập giao diện
interface = gr.Interface(
    fn=gradio_wrapper, 
    inputs=gr.Image(type="pil", label="Tải ảnh lên đây nè"),
    outputs=[
        gr.Textbox(label="Chữ trích xuất được (OCR)"),
        gr.Textbox(label="Bản dịch tiếng Việt"),
        gr.Textbox(label="CLIP đoán ảnh này là:")
    ],
    title="AI Vision Pipeline",
    description="Tải ảnh lên để AI quét chữ, dịch thuật và phân loại nội dung luôn nhé!",
    theme="soft"
)

# Chạy giao diện
interface.launch(share=True)

C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://de4c9cbd8c0a14bb7e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\gradio\blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^